# Write to Bronze Layer
#### CSV -> PySpark -> Parquet -> DuckDB

Steps and actions: 
- Used PERMISSIVE mode with a _corrupt_record column to save malformed rows (according to the transaction schema) without crashing the pipeline.
- Created an index_to_drop field to the Spark schema to catch the unnamed index, then immediately dropped it.
- Used monotonically_increasing_id() to prevent single-node Out-Of-Memory (OOM) crashes.
- Wrote to disk as 4 partitioned Parquet files (repartition(4)). This prevents RAM overflow and allows DuckDB to read the data using multiple CPU threads simultaneously.
- Schema-on-Read: Used DuckDB's CREATE TABLE AS to automatically inherit the strictly typed schema generated by Spark.

In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name, monotonically_increasing_id
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType, DateType, BooleanType
import duckdb as db

In [1]:
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.bindAddress","127.0.0.1") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/25 13:40:52 WARN Utils: Your hostname, LAPTOP-KOAMFQ7J, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/25 13:40:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 13:40:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [27]:
transaction_schema = StructType([
    StructField("index_to_drop", IntegerType(), True),
    StructField("product_id", StringType(), False),
    StructField("product_parent", LongType(), True),
    StructField("product_title", StringType(), True),
    StructField("vine", StringType(), True),
    StructField("verified_purchase", StringType(), True),
    StructField("review_headline", StringType(), True),
    StructField("review_body", StringType(), True),
    StructField("review_date", DateType(), True),
    StructField("marketplace_id", LongType(), True),
    StructField("product_category_id", LongType(), True),
    StructField("label", BooleanType(), True),
    
    # catches rows that fail the schema check
    StructField("_corrupt_record", StringType(), True)
])

#### Handle input

In [ ]:
# input path
raw_csv_path = "../../reviews (copy)/train-1.csv"

# output path
parquet_output_path = "../../data/bronze (raw)/reviews" 

# Create Spark df with and test with transaction schema and add metadata columns
base_df = (spark.read 
    .option("header", "true") 
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(transaction_schema) 
    .csv(raw_csv_path) 
    .drop("index_to_drop")
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", input_file_name())
    .withColumn("_index", monotonically_increasing_id())
)

# split the data into 4 equal chunks (adjust this number based on your dataset size)
(base_df
    .repartition(4) 
    .write
    .mode("overwrite")
    .parquet(parquet_output_path)
)

#### Database Connection

In [30]:
con = db.connect("../../ProjectData.duckdb")

# ensure the schema exists first!
con.execute("CREATE SCHEMA IF NOT EXISTS bronze;")

# ingest the Parquet partitions
con.execute("""
    CREATE OR REPLACE TABLE bronze.reviews_raw AS 
    SELECT * FROM read_parquet('../../data/bronze (raw)/reviews/*.parquet')
""")

print("Total rows:", con.execute("SELECT COUNT(*) FROM bronze.reviews_raw").fetchone()[0])
print("Corrupt rows:", con.execute("SELECT COUNT(*) FROM bronze.reviews_raw WHERE _corrupt_record IS NOT NULL").fetchone()[0])

con.close()

Total rows: 1200
Corrupt rows: 140
